In [1]:
import pkg_resources
import pandas as pd
import numpy as np 
import seaborn as sns
import sklearn
import azure
import matplotlib.pyplot as plt
from azure.ai.ml import MLClient
from azure.ai.ml import command
from azure.identity import DefaultAzureCredential , AzureCliCredential
from azure.core.exceptions import HttpResponseError
import mltable
from sklearn.model_selection import train_test_split
from azure.ai.ml.entities import Data
from azure.ai.ml import automl
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import Input
from azure.ai.ml.entities import Environment
from ucimlrepo import fetch_ucirepo
from azure.identity import AzureCliCredential
from mltable import DataType
from azure.ai.ml.entities import AmlCompute
import mlflow
import subprocess
import yaml
import sys

C:\Users\leste\AppData\Local\Temp\ipykernel_29356\2473914065.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
c:\Users\leste\AppData\Local\PythonEnvs\MLOps-Python-3.12.14\Lib\site-packages\mlflow\__init__.py:41: UserWarning: Versions of mlflow (3.18.0) and child packages mlflow-skinny (3.15.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()
c:\Users\leste\AppData\Local\PythonEnvs\MLOps-Python-3.12.14\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
subprocess.run("azd auth login")

CompletedProcess(args='azd auth login', returncode=0)

In [3]:
# Connecting to Azure CLI 

my_credential = AzureCliCredential(
    subscription= "48f83b14-32a8-4911-b629-ea5aead7fb29"
)

token = my_credential.get_token(
    "https://management.azure.com/.default"
)

In [4]:
# Connecting to Azure ML
ml_client = MLClient(
    credential = my_credential,
    subscription_id = "48f83b14-32a8-4911-b629-ea5aead7fb29",
    resource_group_name =  "lm1610-rg",
    workspace_name =  "lm_portfolio",
)

In [5]:
# Creating a compute cluster
aml_compute = AmlCompute(
    name = 'aml-compute-cluster-v1',
    description= 'This cluster is created to run my basic-data-split package',
    size = "STANDARD_DS3_V2",
    min_instances= 0,
    max_instances= 2,
    idle_time_before_scale_down= 120
)

In [6]:
compute_creation = ml_client.compute.begin_create_or_update(aml_compute)
compute_creation.result()

AmlCompute({'type': 'amlcompute', 'created_on': '2026-11-26T10:01:08.2765011+00:00', 'provisioning_state': <ProvisioningState.SUCCEEDED: 'Succeeded'>, 'provisioning_errors': None, 'name': 'aml-compute-cluster-v1', 'description': 'This cluster is created to run my basic-data-split package', 'tags': None, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/48f83b14-32a8-4911-b629-ea5aead7fb29/resourceGroups/lm1610-rg/providers/Microsoft.MachineLearningServices/workspaces/lm_portfolio/computes/aml-compute-cluster-v1', 'Resource__source_path': '', 'base_path': 'c:\\Users\\leste\\OneDrive\\Desktop\\Microsoft VS Code\\Github\\MLOps-Portfolio\\MLOps Notebooks', 'creation_context': None, 'resource_id': None, 'location': 'centralindia', 'size': 'STANDARD_DS3_V2', 'min_instances': 0, 'max_instances': 2, 'idle_time_before_scale_down': 120.0, 'identity': None, 'ssh_public_access_enabled': True, 'ssh_settings': None, 'network_settings': <azure.ai.ml.entities._compute.compute.NetworkSett

In [8]:
for data in ml_client.data.list():
    print(data.name)

Bank-MLtable
dataset


In [18]:
data_input = Data(
    type = AssetTypes.URI_FILE,
    path = "C:/Users/leste/OneDrive/Desktop/Microsoft VS Code/Github/MLOps-Portfolio/MLOps py files/bank-marketing.csv",
    description = "This dataset is created to test the command job",
    name = "Bank_Data_csv",
    version = "v1",
    )

In [19]:
ml_client.data.create_or_update(data_input)

Data({'path': 'azureml://subscriptions/48f83b14-32a8-4911-b629-ea5aead7fb29/resourcegroups/lm1610-rg/workspaces/lm_portfolio/datastores/workspaceblobstore/paths/LocalUpload/bd6e69f04fea4addafe86584f13caf2f4a859babfb4578a9c43916b90d954ae4/bank-marketing.csv', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_file', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'Bank_Data_csv', 'description': 'This dataset is created to test the command job', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/48f83b14-32a8-4911-b629-ea5aead7fb29/resourceGroups/lm1610-rg/providers/Microsoft.MachineLearningServices/workspaces/lm_portfolio/data/Bank_Data_csv/versions/v1', 'Resource__source_path': '', 'base_path': 'c:\\Users\\leste\\OneDrive\\Desktop\\Microsoft VS Code\\Github\\MLOps-Portfolio\\MLOps Notebooks', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x0000022

In [43]:
command_inputs = {
    "my_input" : Input(
        type = AssetTypes.URI_FILE,
        path = "azureml:Bank_Data_csv:v1"
    ),
    "test_size" : 0.3,
    "random_state" : 42
}

In [30]:
for env in ml_client.environments.list():
    print(env.name)

mlops-local-environment
AzureML-AI-Studio-Development
AzureML-ACPT-pytorch-1.13-py38-cuda11.7-gpu
AzureML-ACPT-pytorch-1.12-py38-cuda11.6-gpu
AzureML-ACPT-pytorch-1.12-py39-cuda11.6-gpu
AzureML-ACPT-pytorch-1.11-py38-cuda11.5-gpu
AzureML-ACPT-pytorch-1.11-py38-cuda11.3-gpu
AzureML-responsibleai-0.21-ubuntu20.04-py38-cpu
AzureML-responsibleai-0.20-ubuntu20.04-py38-cpu
AzureML-tensorflow-2.5-ubuntu20.04-py38-cuda11-gpu
AzureML-tensorflow-2.6-ubuntu20.04-py38-cuda11-gpu
AzureML-tensorflow-2.7-ubuntu20.04-py38-cuda11-gpu
AzureML-sklearn-1.0-ubuntu20.04-py38-cpu
AzureML-pytorch-1.10-ubuntu18.04-py38-cuda11-gpu
AzureML-pytorch-1.9-ubuntu18.04-py37-cuda11-gpu
AzureML-pytorch-1.8-ubuntu18.04-py37-cuda11-gpu
AzureML-sklearn-0.24-ubuntu18.04-py37-cpu
AzureML-lightgbm-3.2-ubuntu18.04-py37-cpu
AzureML-pytorch-1.7-ubuntu18.04-py37-cuda11-gpu
AzureML-tensorflow-2.4-ubuntu18.04-py37-cuda11-gpu
AzureML-Triton
AzureML-Designer-Score
AzureML-PyTorch-1.3-CPU
AzureML-VowpalWabbit-8.8.0


In [ ]:
command_job = command(
    code = "C:/Users/leste/OneDrive/Desktop/Microsoft VS Code/Github/MLOps-Portfolio/MLOps py files",
    inputs= command_inputs,
    command = "python basic-data-split.py --input_data ${{inputs.my_input}} --test_size ${{inputs.test_size}} --random_state ${{inputs.random_state}} ", 
    description="This command is created to run the basic-data-split py file explicitly",
    environment = "AzureML-sklearn-0.24-ubuntu18.04-py37-cpu@latest",
    compute = 'aml-compute-cluster-v1',
    experiment_name= 'First-comprehensive-command-job',
    display_name = "Basic-split-job",
    
)

In [45]:
ml_client.jobs.create_or_update(command_job)

Experiment,Name,Type,Status,Details Page
First-comprehensive-command-job,amiable_rod_j5xkb4xzrh,command,JobStatus.STARTING,Link to Azure Machine Learning studio


In [53]:
ml_client.jobs.get("amiable_rod_j5xkb4xzrh")

Experiment,Name,Type,Status,Details Page
First-comprehensive-command-job,amiable_rod_j5xkb4xzrh,command,JobStatus.COMPLETED,Link to Azure Machine Learning studio


In [54]:
# Deleting the compute cluster once the output is retrieved 

ml_client.compute.begin_delete('aml-compute-cluster-v1')